# 3.1 一种无需训练权重的简单自注意力机制

## **步骤 1：** 计算未归一化的注意力分数 $\omega$

### 假设我们有如下输入句子，它已经按照第 3 章所述嵌入到 3 维向量中（为了便于说明，这里我们使用了一个非常小的嵌入维度，以便它可以完整地显示在页面上而无需换行）：

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

### (本书遵循机器学习和深度学习的常用约定，将训练样本表示为行，特征值表示为列；以上述张量为例，每一行代表一个词，每一列代表一个嵌入维度。)

In [2]:
query = inputs[1] # 第二个输入词元
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # 点积（由于它们是一维向量，因此无需转置）
print(attn_scores_2) # 第 2 个 token 的 注意力得分，点积序列

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### 附注：点积本质上是将两个向量逐元素相乘，然后将结果相加的简写形式：

In [3]:
res = 0.
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]
print(res) # 第二个词向量 与 第一个词向量的点积
print(torch.dot(inputs[0], query)) # torch.dot 的 方法一样的

tensor(0.9544)
tensor(0.9544)


## **步骤 2：** 将未归一化的注意力得分 $\omega$ 归一化，使其总和为 1

以下是一种将未归一化的注意力分数归一化为总和为 1 的简单方法（这是一种约定俗成的做法，有利于解释，并且对训练稳定性很重要）：

In [4]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("注意力权重: ", attn_weights_2_tmp)
print("注意力分数总和: ", attn_weights_2_tmp.sum())

注意力权重:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
注意力分数总和:  tensor(1.0000)


### 然而，在实践中推荐使用 softmax 函数，因为它更擅长处理极端值，并且在训练过程中具有更理想的梯度特性。
- 以下是一个简单的 softmax 函数实现，用于缩放，它还会对向量元素进行归一化，使它们的总和为 1：

In [5]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("注意力权重: ", attn_weights_2_naive)
print("注意力分数总和: ", attn_weights_2_naive.sum())

注意力权重:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
注意力分数总和:  tensor(1.)


- 上述简单实现方法对于过大或过小的输入值都可能由于溢出和下溢问题而出现数值不稳定问题。

### 使用 PyTorch 实现的 softmax 函数，该函数针对性能进行了高度优化：

In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim = 0)
print("注意力权重: ", attn_weights_2)
print("注意力分数总和: ", attn_weights_2.sum())

注意力权重:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
注意力分数总和:  tensor(1.)


## **步骤 3** ：通过将嵌入的输入标记 $x^{(i)}$ 与注意力权重相乘来计算上下文向量 $z^{(2)}$，并将结果向量相加：

In [7]:
query = inputs[1] # 第二个输入词元
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## 3.1.1 计算所有输入词元的注意力权重

#### 将前面的步骤 1 应用于所有成对元素，以计算未归一化的注意力得分矩阵：

In [8]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### 我们可以通过矩阵乘法更高效地实现上述目标：

In [9]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


#### 与之前的步骤 2 类似，我们对每一行进行归一化，使每一行的值之和为 1：

In [10]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- 快速验证每一行中的值之和是否确实为 1：

In [11]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("第二行元素总和: ", row_2_sum)

print("全部行的各总和: ", attn_weights.sum(dim=-1))

第二行元素总和:  1.0
全部行的各总和:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


#### 应用之前的步骤 3 来计算所有上下文向量：

In [12]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- #### 为了进行健全性检查，先前计算的上下文向量 $z^{(2)}=[0.4419, 0.6515, 0.5683]$ 可以在上面的第 2 行中找到：

---

# 3.2 实现具有可训练权重的自注意力机制

- ### 在 GPT 模型中，输入和输出维度通常是相同的，但为了便于说明，更好地跟踪计算过程，我们在这里选择不同的输入和输出维度：

In [13]:
x_2 = inputs[1] # 第二输入元素
d_in = inputs.shape[1] # 输入嵌入的大小，d=3
d_out = 2 # 输出嵌入大小，d=2

- #### 下面，我们初始化三个权重矩阵；请注意，为了便于说明，我们将 `requires_grad=False` 以减少输出中的冗余信息。但如果要将这些权重矩阵用于模型训练，则需要将 `requires_grad=True` 在模型训练期间更新这些矩阵。

In [15]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 接下来，我们计算`查询向量`、`键向量`和`值向量`：

In [17]:
query_2 = x_2 @ W_query # _2 因为它与第二个输入元素有关
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


- 我们可以通过矩阵乘法获得所有键和值（将 6 个输入词元从 3D 空间投影到 2D 嵌入空间）:

In [20]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


## **步骤 1**：计算查询向量与每个键向量之间的 `点积` 来计算未归一化的`注意力得分`： 

- 首先，让我们计算未归一化的注意力得分$w_{22}$

In [22]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


- 同样，我们可以通过矩阵乘法将此计算推广到所有注意力得分$w$：

In [25]:
attn_scores_2 = query_2 @ keys.T # 给定查询的所有注意力得分
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


## **步骤 2**：我们使用之前用过的 softmax 函数计算 `注意力权重`（归一化的注意力得分之和为 1）。

- 与之前不同的是，我们现在通过将 `注意力得分` 除以嵌入维度的平方根 $\sqrt{d_k}$ 即 （`d_k**0.5` ）来缩放`注意力得分`：

In [26]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


## **步骤 3**：计算输入查询向量 2 的`上下文向量`：

In [27]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])
